## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='processing-2.processing.data.production.internal' ,
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [6]:
## base

base_query = f"""

with base as (
    select 
        customer_id
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-03-17'
        and taxi_lifetime_last_ride_city = 'Bangalore'
    ),
    
    active_week as (
    
    select
        customerid,
        91 max_active_days,
        sum(rr_active_day) total_active_days,
        sum(case when week_start_date = '20241216' and rr_active_day > 0 then 1 else 0 end ) isAct_w01,
        sum(case when week_start_date = '20241223' and rr_active_day > 0 then 1 else 0 end ) isAct_w02,
        sum(case when week_start_date = '20241230' and rr_active_day > 0 then 1 else 0 end ) isAct_w03,
        sum(case when week_start_date = '20250106' and rr_active_day > 0 then 1 else 0 end ) isAct_w04,
        sum(case when week_start_date = '20250113' and rr_active_day > 0 then 1 else 0 end ) isAct_w05,
        sum(case when week_start_date = '20250120' and rr_active_day > 0 then 1 else 0 end ) isAct_w06,
        sum(case when week_start_date = '20250127' and rr_active_day > 0 then 1 else 0 end ) isAct_w07,
        sum(case when week_start_date = '20250203' and rr_active_day > 0 then 1 else 0 end ) isAct_w08,
        sum(case when week_start_date = '20250210' and rr_active_day > 0 then 1 else 0 end ) isAct_w09,
        sum(case when week_start_date = '20250217' and rr_active_day > 0 then 1 else 0 end ) isAct_w10,
        sum(case when week_start_date = '20250224' and rr_active_day > 0 then 1 else 0 end ) isAct_w11,
        sum(case when week_start_date = '20250303' and rr_active_day > 0 then 1 else 0 end ) isAct_w12,
        sum(case when week_start_date = '20250310' and rr_active_day > 0 then 1 else 0 end ) isAct_w13,
        
        13 max_active_weeks,
        count(distinct case when weekly_active_day > 0 then week_start_date end) total_active_weeks,
        sum(case when week_start_date = '20241216' then weekly_active_day end ) act_days_w01,
        sum(case when week_start_date = '20241223' then weekly_active_day end ) act_days_w02,
        sum(case when week_start_date = '20241230' then weekly_active_day end ) act_days_w03,
        sum(case when week_start_date = '20250106' then weekly_active_day end ) act_days_w04,
        sum(case when week_start_date = '20250113' then weekly_active_day end ) act_days_w05,
        sum(case when week_start_date = '20250120' then weekly_active_day end ) act_days_w06,
        sum(case when week_start_date = '20250127' then weekly_active_day end ) act_days_w07,
        sum(case when week_start_date = '20250203' then weekly_active_day end ) act_days_w08,
        sum(case when week_start_date = '20250210' then weekly_active_day end ) act_days_w09,
        sum(case when week_start_date = '20250217' then weekly_active_day end ) act_days_w10,
        sum(case when week_start_date = '20250224' then weekly_active_day end ) act_days_w11,
        sum(case when week_start_date = '20250303' then weekly_active_day end ) act_days_w12,
        sum(case when week_start_date = '20250310' then weekly_active_day end ) act_days_w13
        
        
    from
    (
        select
            date_format(date_trunc('week', date_parse(day, '%Y-%m-%d')), '%Y%m%d') week_start_date,
            customerid,
            count(distinct case when rr_active_day > 0 then day end) weekly_active_day,
            sum(rr_active_day) rr_active_day
        from
        (
            select 
                day,
                customerid, 
                -- max(fe_sessions_unique_daily) fe_session, 
                sum(rr_sessions_unique_daily) rr_session, 
                max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_active_day
            from 
                datasets.customer_rf_daily_kpi
            where 
                day >= '2024-12-16'
                and day <= '2025-03-16'
                and city = 'Bangalore'
                and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
                -- and customerid = '62a16eb57d7967a5a8e2b69c'
            group by 1,2
            order by 1
        )
        group by 1,2
    )
    group by 1

    )
    
    select
        *
    from
        base
    join 
        active_week
        on base.customer_id = active_week.customerid
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

,customer_id,customerid,max_active_days,total_active_days,isAct_w01,isAct_w02,isAct_w03,isAct_w04,isAct_w05,isAct_w06,isAct_w07,isAct_w08,isAct_w09,isAct_w10,isAct_w11,isAct_w12,isAct_w13,max_active_weeks,total_active_weeks,act_days_w01,act_days_w02,act_days_w03,act_days_w04,act_days_w05,act_days_w06,act_days_w07,act_days_w08,act_days_w09,act_days_w10,act_days_w11,act_days_w12,act_days_w13
0,5e42fe240d9a1c5d5a30f5c6,5e42fe240d9a1c5d5a30f5c6,91,1,0,0,0,0,0,0,0,0,0,1,0,0,0,13,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN
1,6627f5777188cc4faf98c9a0,6627f5777188cc4faf98c9a0,91,48,1,1,1,1,1,1,0,1,1,1,1,1,1,13,12,3.0,2.0,7.0,4.0,4.0,3.0,NaN,3.0,2.0,3.0,4.0,6.0,7.0
2,64ff31888e9d1eee7964ea6c,64ff31888e9d1eee7964ea6c,91,14,1,1,1,1,1,0,1,0,1,0,0,0,1,13,8,3.0,3.0,1.0,2.0,1.0,NaN,1.0,NaN,1.0,0.0,0.0,0.0,2.0
3,5f26d60aea5a8becf57c2bbd,5f26d60aea5a8becf57c2bbd,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,5f7804397d99a75f7059c89b,5f7804397d99a75f7059c89b,91,3,1,0,0,0,0,0,1,0,0,0,0,0,0,13,2,1.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4929720,639c6c3f9665f5917c8d00e7,639c6c3f9665f5917c8d00e7,91,1,0,0,0,0,1,0,0,0,0,0,0,0,0,13,1,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4929721,61b9556af3de6f2c234d05e1,61b9556af3de6f2c234d05e1,91,1,0,0,1,0,0,0,0,0,0,0,0,0,0,13,1,NaN,0.0,1.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4929722,62ac147a17af9aeebf59ff95,62ac147a17af9aeebf59ff95,91,7,1,0,0,0,1,0,0,0,0,0,1,0,0,13,3,4.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
4929723,64ad788d5e002d33298e310f,64ad788d5e002d33298e310f,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [4]:
df_base_query = pd.read_parquet('data_dump_for_process_mining_bangalore.parquet')

In [5]:
df_base_query.shape

(4929725, 32)

In [6]:
df_base_query.head(5)

,customer_id,customerid,max_active_days,total_active_days,isAct_w01,isAct_w02,isAct_w03,isAct_w04,isAct_w05,isAct_w06,isAct_w07,isAct_w08,isAct_w09,isAct_w10,isAct_w11,isAct_w12,isAct_w13,max_active_weeks,total_active_weeks,act_days_w01,act_days_w02,act_days_w03,act_days_w04,act_days_w05,act_days_w06,act_days_w07,act_days_w08,act_days_w09,act_days_w10,act_days_w11,act_days_w12,act_days_w13
0,5e42fe240d9a1c5d5a30f5c6,5e42fe240d9a1c5d5a30f5c6,91,1,0,0,0,0,0,0,0,0,0,1,0,0,0,13,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN
1,6627f5777188cc4faf98c9a0,6627f5777188cc4faf98c9a0,91,48,1,1,1,1,1,1,0,1,1,1,1,1,1,13,12,3.0,2.0,7.0,4.0,4.0,3.0,NaN,3.0,2.0,3.0,4.0,6.0,7.0
2,64ff31888e9d1eee7964ea6c,64ff31888e9d1eee7964ea6c,91,14,1,1,1,1,1,0,1,0,1,0,0,0,1,13,8,3.0,3.0,1.0,2.0,1.0,NaN,1.0,NaN,1.0,0.0,0.0,0.0,2.0
3,5f26d60aea5a8becf57c2bbd,5f26d60aea5a8becf57c2bbd,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,5f7804397d99a75f7059c89b,5f7804397d99a75f7059c89b,91,3,1,0,0,0,0,0,1,0,0,0,0,0,0,13,2,1.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# df_base_query.to_parquet('data_dump_for_process_mining_bangalore.parquet', index=False)
# df_base_query.to_csv('data_dump_for_process_mining_bangalore.csv', index=False)

## Data Engineering

#### Adding required active month fields 

In [8]:
df_base_query['isAct_m01'] = np.where(df_base_query[['isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04']].sum(axis=1) >= 1, 1, 0)
df_base_query['isAct_m02'] = np.where(df_base_query[['isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08']].sum(axis=1) >= 1, 1, 0)
df_base_query['isAct_m03'] = np.where(df_base_query[['isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13']].sum(axis=1) >= 1, 1, 0)

df_base_query['max_active_months'] = 3
df_base_query['total_active_months'] = df_base_query[['isAct_m01', 'isAct_m02', 'isAct_m03']].sum(axis=1)

In [9]:
df_base_query['additional_filter_21days'] = np.where(df_base_query[['isAct_w11', 'isAct_w12', 'isAct_w13']].sum(axis=1) >= 1, 1, 0)

#### Adding required derived percentage fields for making thresholds

In [10]:
df_base_query[['customerid', 
               'max_active_days', 'total_active_days', 
               'max_active_weeks', 'total_active_weeks', 
               'max_active_months', 'total_active_months']].head(5)

,customerid,max_active_days,total_active_days,max_active_weeks,total_active_weeks,max_active_months,total_active_months
0,5e42fe240d9a1c5d5a30f5c6,91,1,13,1,3,1
1,6627f5777188cc4faf98c9a0,91,48,13,12,3,3
2,64ff31888e9d1eee7964ea6c,91,14,13,8,3,3
3,5f26d60aea5a8becf57c2bbd,91,0,13,0,3,0
4,5f7804397d99a75f7059c89b,91,3,13,2,3,2


In [11]:
df_base_query['active_days_percet'] = (df_base_query['total_active_days']/df_base_query['max_active_days']).round(2)
df_base_query['active_weeks_percet'] = (df_base_query['total_active_weeks']/df_base_query['max_active_weeks']).round(2)
df_base_query['active_months_percet'] = (df_base_query['total_active_months']/df_base_query['max_active_months']).round(2)

In [12]:
df_base_query[['customerid', 
               'max_active_days', 'total_active_days', 'active_days_percet', 
               'max_active_weeks', 'total_active_weeks', 'active_weeks_percet', 
               'max_active_months', 'total_active_months', 'active_months_percet']].head(5)

,customerid,max_active_days,total_active_days,active_days_percet,max_active_weeks,total_active_weeks,active_weeks_percet,max_active_months,total_active_months,active_months_percet
0,5e42fe240d9a1c5d5a30f5c6,91,1,0.01,13,1,0.08,3,1,0.33
1,6627f5777188cc4faf98c9a0,91,48,0.53,13,12,0.92,3,3,1.00
2,64ff31888e9d1eee7964ea6c,91,14,0.15,13,8,0.62,3,3,1.00
3,5f26d60aea5a8becf57c2bbd,91,0,0.00,13,0,0.00,3,0,0.00
4,5f7804397d99a75f7059c89b,91,3,0.03,13,2,0.15,3,2,0.67


#### Segment Tagging

In [13]:
df_base_query['daily_segment'] = np.where(df_base_query['active_days_percet'] >= 0.45, '1-DAILY', None)

df_base_query['weekly_segment'] = np.where((df_base_query['active_weeks_percet'] >= 0.70) 
                                           & 
                                           (df_base_query['active_days_percet'] < 0.45), '2-WEEKLY', None)


df_base_query['monthly_segment'] = np.where((df_base_query['active_months_percet'] >= 0.66) 
                                            &
                                            (df_base_query['active_weeks_percet'] < 0.70) 
                                            & 
                                            (df_base_query['active_days_percet'] < 0.45), '3-MONTHLY', None)

df_base_query['inactive_segment'] = np.where(df_base_query['total_active_days'] == 0, '5-INACTIVE', None)

df_base_query['segment'] = df_base_query['daily_segment']\
                                .fillna(df_base_query['weekly_segment'])\
                                .fillna(df_base_query['monthly_segment'])\
                                .fillna(df_base_query['inactive_segment'])\
                                .fillna('4-OTHER')

In [14]:
df_base_query[['customerid', 
               'active_days_percet', 'active_weeks_percet', 'active_months_percet', 
               'daily_segment', 'weekly_segment', 'monthly_segment', 'inactive_segment',
               'segment']].head(5)

,customerid,active_days_percet,active_weeks_percet,active_months_percet,daily_segment,weekly_segment,monthly_segment,inactive_segment,segment
0,5e42fe240d9a1c5d5a30f5c6,0.01,0.08,0.33,None,None,None,None,4-OTHER
1,6627f5777188cc4faf98c9a0,0.53,0.92,1.00,1-DAILY,None,None,None,1-DAILY
2,64ff31888e9d1eee7964ea6c,0.15,0.62,1.00,None,None,3-MONTHLY,None,3-MONTHLY
3,5f26d60aea5a8becf57c2bbd,0.00,0.00,0.00,None,None,None,5-INACTIVE,5-INACTIVE
4,5f7804397d99a75f7059c89b,0.03,0.15,0.67,None,None,3-MONTHLY,None,3-MONTHLY


In [15]:
df_base_query.shape

(4929725, 46)

#### Distribution

In [16]:
# 45%
df_base_query.groupby('segment').agg(customers = ('customerid', 'nunique')).sort_values(by='segment', ascending=True).reset_index()

,segment,customers
0,1-DAILY,108901
1,2-WEEKLY,296457
2,3-MONTHLY,2172070
3,4-OTHER,1796495
4,5-INACTIVE,555802


In [17]:
# 45%
df_base_query[~df_base_query['segment'].isin(['5-INACTIVE'])]['segment'].value_counts(normalize=True)

segment
3-MONTHLY    0.496595
4-OTHER      0.410729
2-WEEKLY     0.067778
1-DAILY      0.024898
Name: proportion, dtype: float64

In [18]:
# 45%
df_base_query['segment'].value_counts(normalize=True)

segment
3-MONTHLY     0.440607
4-OTHER       0.364421
5-INACTIVE    0.112745
2-WEEKLY      0.060137
1-DAILY       0.022091
Name: proportion, dtype: float64

In [19]:
df_base_query[['customer_id', 'segment',    
               
               'max_active_days', 'total_active_days', 'active_days_percet', 'daily_segment',
               'isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04', 'isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08', 'isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13', 
               
               'max_active_weeks', 'total_active_weeks', 'active_weeks_percet', 'weekly_segment',
               'act_days_w01', 'act_days_w02', 'act_days_w03', 'act_days_w04', 'act_days_w05', 'act_days_w06', 'act_days_w07', 'act_days_w08', 'act_days_w09', 'act_days_w10', 'act_days_w11', 'act_days_w12', 'act_days_w13', 
               
               'max_active_months', 'total_active_months', 'active_months_percet', 'monthly_segment',
               'isAct_m01', 'isAct_m02', 'isAct_m03', 'inactive_segment']]

,customer_id,segment,max_active_days,total_active_days,active_days_percet,daily_segment,isAct_w01,isAct_w02,isAct_w03,isAct_w04,isAct_w05,isAct_w06,isAct_w07,isAct_w08,isAct_w09,isAct_w10,isAct_w11,isAct_w12,isAct_w13,max_active_weeks,total_active_weeks,active_weeks_percet,weekly_segment,act_days_w01,act_days_w02,act_days_w03,act_days_w04,act_days_w05,act_days_w06,act_days_w07,act_days_w08,act_days_w09,act_days_w10,act_days_w11,act_days_w12,act_days_w13,max_active_months,total_active_months,active_months_percet,monthly_segment,isAct_m01,isAct_m02,isAct_m03,inactive_segment
0,5e42fe240d9a1c5d5a30f5c6,4-OTHER,91,1,0.01,None,0,0,0,0,0,0,0,0,0,1,0,0,0,13,1,0.08,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN,3,1,0.33,None,0,0,1,None
1,6627f5777188cc4faf98c9a0,1-DAILY,91,48,0.53,1-DAILY,1,1,1,1,1,1,0,1,1,1,1,1,1,13,12,0.92,None,3.0,2.0,7.0,4.0,4.0,3.0,NaN,3.0,2.0,3.0,4.0,6.0,7.0,3,3,1.00,None,1,1,1,None
2,64ff31888e9d1eee7964ea6c,3-MONTHLY,91,14,0.15,None,1,1,1,1,1,0,1,0,1,0,0,0,1,13,8,0.62,None,3.0,3.0,1.0,2.0,1.0,NaN,1.0,NaN,1.0,0.0,0.0,0.0,2.0,3,3,1.00,3-MONTHLY,1,1,1,None
3,5f26d60aea5a8becf57c2bbd,5-INACTIVE,91,0,0.00,None,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,0.00,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3,0,0.00,None,0,0,0,5-INACTIVE
4,5f7804397d99a75f7059c89b,3-MONTHLY,91,3,0.03,None,1,0,0,0,0,0,1,0,0,0,0,0,0,13,2,0.15,None,1.0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,3,2,0.67,3-MONTHLY,1,1,0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4929720,639c6c3f9665f5917c8d00e7,4-OTHER,91,1,0.01,None,0,0,0,0,1,0,0,0,0,0,0,0,0,13,1,0.08,None,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,1,0.33,None,0,1,0,None
4929721,61b9556af3de6f2c234d05e1,4-OTHER,91,1,0.01,None,0,0,1,0,0,0,0,0,0,0,0,0,0,13,1,0.08,None,NaN,0.0,1.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,3,1,0.33,None,1,0,0,None
4929722,62ac147a17af9aeebf59ff95,3-MONTHLY,91,7,0.08,None,1,0,0,0,1,0,0,0,0,0,1,0,0,13,3,0.23,None,4.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,3,3,1.00,3-MONTHLY,1,1,1,None
4929723,64ad788d5e002d33298e310f,5-INACTIVE,91,0,0.00,None,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,0.00,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,3,0,0.00,None,0,0,0,5-INACTIVE


In [20]:
df_base_query[['customer_id', 'segment',    
               
               'max_active_days', 'total_active_days', 'active_days_percet', 'daily_segment',
               'isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04', 'isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08', 'isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13', 
               
               'max_active_weeks', 'total_active_weeks', 'active_weeks_percet', 'weekly_segment',
               'act_days_w01', 'act_days_w02', 'act_days_w03', 'act_days_w04', 'act_days_w05', 'act_days_w06', 'act_days_w07', 'act_days_w08', 'act_days_w09', 'act_days_w10', 'act_days_w11', 'act_days_w12', 'act_days_w13', 
               
               'max_active_months', 'total_active_months', 'active_months_percet', 'monthly_segment',
               'isAct_m01', 'isAct_m02', 'isAct_m03', 'inactive_segment']].to_csv('new_definition_regularity_segment_v2_45perc.csv', header=False, index=False)

In [48]:
df_base_query[df_base_query['total_active_days'] == 0]

,customer_id,customerid,max_active_days,total_active_days,isAct_w01,isAct_w02,isAct_w03,isAct_w04,isAct_w05,isAct_w06,isAct_w07,isAct_w08,isAct_w09,isAct_w10,isAct_w11,isAct_w12,isAct_w13,max_active_weeks,total_active_weeks,act_days_w01,act_days_w02,act_days_w03,act_days_w04,act_days_w05,act_days_w06,act_days_w07,act_days_w08,act_days_w09,act_days_w10,act_days_w11,act_days_w12,act_days_w13,isAct_m01,isAct_m02,isAct_m03,max_active_months,total_active_months,additional_filter_21days,active_days_percet,active_weeks_percet,active_months_percet,daily_segment,weekly_segment,monthly_segment,inactive_segment,segment
3,5f26d60aea5a8becf57c2bbd,5f26d60aea5a8becf57c2bbd,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
14,628babf6a0c1d7a3d2df7a30,628babf6a0c1d7a3d2df7a30,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
19,678d10ad6bfaa817d8fe3af9,678d10ad6bfaa817d8fe3af9,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
26,6043fd5d2b32d21385cf7b0a,6043fd5d2b32d21385cf7b0a,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
35,630dfcb75843ea2182b5a9c7,630dfcb75843ea2182b5a9c7,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4929701,6012406671432314b0c3b7d6,6012406671432314b0c3b7d6,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
4929704,63ac3f7ac56b51707bf22183,63ac3f7ac56b51707bf22183,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
4929709,61cfd2459088371df6e31517,61cfd2459088371df6e31517,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
4929716,619e8c230c6dce5a4d16f803,619e8c230c6dce5a4d16f803,91,0,0,0,0,0,0,0,0,0,0,0,0,0,0,13,0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3,0,0,0.0,0.0,0.0,None,None,None,5-INACTIVE,5-INACTIVE
